# Results Review

Review validation candidates, held-out test metrics, and representative evaluation figures from the current Halfmile run.

In [ ]:
from pathlib import Path
import json
import sys
from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt
import pandas as pd

from _notebook_setup import PROCESSED_DIR, REPORTS_DIR, require_path

%matplotlib inline
plt.style.use("default")

This setup cell brings in the tools for reading saved reports, plotting summaries, and displaying existing figure files. It keeps the notebook squarely focused on review rather than regeneration of results. In other words, we are opening the current state of the experiment, not rerunning the whole pipeline.

In [ ]:
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "seismic_first_break_picker").exists():
    repo_root = repo_root.parent
if not (repo_root / "seismic_first_break_picker").exists():
    raise RuntimeError("Could not locate the repository root. Start Jupyter from anywhere inside this repo.")
for path in (repo_root, repo_root / "notebooks"):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

This cell makes the notebook resilient to where the kernel started and ensures that local helper modules resolve cleanly. That sounds mundane, but it removes a lot of notebook fragility. The outcome is that every report path and import below are anchored to the repository instead of the user's current directory.

## Load Saved Reports

The next cell reads the current training report, held-out evaluation summary, and per-segment CSV outputs. That gives the notebook a single in-memory view of both model-selection metrics and final test-set reporting.

In [ ]:
training_report_path = require_path(PROCESSED_DIR / "ml_correction_training_report.json")
summary_path = require_path(REPORTS_DIR / "evaluation" / "test_summary.json")
summary_table_path = require_path(REPORTS_DIR / "evaluation" / "test_summary_table.csv")
segment_metrics_path = require_path(REPORTS_DIR / "evaluation" / "test_segment_metrics.csv")

with open(training_report_path, "r", encoding="utf-8") as handle:
    training_report = json.load(handle)
with open(summary_path, "r", encoding="utf-8") as handle:
    test_summary = json.load(handle)

summary_table = pd.read_csv(summary_table_path)
segment_metrics = pd.read_csv(segment_metrics_path)

display(summary_table.round(3))
display(pd.DataFrame([training_report["selected_config"]]))
pd.DataFrame([test_summary["improvement"]]).round(3)

Here we pull the training report, aggregate evaluation summary, and per-segment metrics into one place. That gives us both the broad test-set result and the configuration that produced it. It is the fastest way to remind ourselves what the current run achieved before digging into candidate-by-candidate details.

## Validation Candidate Comparison

This section normalizes the candidate grid into a dataframe and plots the main validation metrics. The intent is to make model selection transparent: which configuration won, and by how much.

In [ ]:
candidate_df = pd.json_normalize(training_report["candidate_results"])
candidate_df = candidate_df.rename(
    columns={
        "validation_metrics.corrected_mae_ms": "corrected_mae_ms",
        "validation_metrics.corrected_rmse_ms": "corrected_rmse_ms",
        "validation_metrics.mae_improvement_ms": "mae_improvement_ms",
        "config.learning_rate": "learning_rate",
        "config.max_iter": "max_iter",
        "config.max_depth": "max_depth",
    }
)

display(candidate_df[["candidate_index", "corrected_mae_ms", "corrected_rmse_ms", "mae_improvement_ms", "learning_rate", "max_iter", "max_depth"]].round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].bar(candidate_df["candidate_index"].astype(str), candidate_df["corrected_mae_ms"], color="tab:blue")
axes[0].set_title("Validation corrected MAE")
axes[0].set_xlabel("Candidate")
axes[0].set_ylabel("MAE (ms)")

axes[1].bar(candidate_df["candidate_index"].astype(str), candidate_df["corrected_rmse_ms"], color="tab:orange")
axes[1].set_title("Validation corrected RMSE")
axes[1].set_xlabel("Candidate")
axes[1].set_ylabel("RMSE (ms)")
plt.show()

This section makes model selection legible. Flattening the candidate results into a dataframe and plotting the validation MAE and RMSE lets us see whether the chosen configuration won decisively or only narrowly. That context is useful when deciding whether the current hyperparameter search was good enough or worth extending.

## Segment-Level Behavior

Aggregate MAE and RMSE are useful, but they can hide how consistently the method helps. The next cell counts improved versus worsened segments, then plots the distribution of segment-wise MAE gain and the baseline-versus-corrected scatter.

In [ ]:
improved = int((segment_metrics["mae_improvement_ms"] > 0).sum())
worsened = int((segment_metrics["mae_improvement_ms"] < 0).sum())
unchanged = int((segment_metrics["mae_improvement_ms"] == 0).sum())

print(f"Improved segments: {improved}")
print(f"Worsened segments: {worsened}")
print(f"Unchanged segments: {unchanged}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

axes[0].hist(segment_metrics["mae_improvement_ms"], bins=40, color="tab:green", alpha=0.85)
axes[0].axvline(0.0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Distribution of segment MAE improvement")
axes[0].set_xlabel("Baseline MAE - corrected MAE (ms)")
axes[0].set_ylabel("Segment count")

axes[1].scatter(segment_metrics["baseline_mae_ms"], segment_metrics["corrected_mae_ms"], s=12, alpha=0.5, color="tab:purple")
limit = max(segment_metrics["baseline_mae_ms"].max(), segment_metrics["corrected_mae_ms"].max())
axes[1].plot([0, limit], [0, limit], color="black", linestyle="--", linewidth=1)
axes[1].set_title("Per-segment MAE")
axes[1].set_xlabel("Baseline MAE (ms)")
axes[1].set_ylabel("Corrected MAE (ms)")

plt.show()

The aggregate summary tells us the average outcome, but this cell tells us how consistent that outcome was across the held-out set. Counting improved versus worsened segments and plotting the MAE gain distribution makes it much easier to judge robustness. The scatter plot is especially helpful because it shows whether corrected segments mostly fall below the baseline line, which is exactly what we want.

## Representative Figure Gallery

The final cell opens the saved evaluation images for the best, median, worst, and regression cases. These are the same artifacts referenced from the report and presentation material, so they give a direct visual readout of the current pipeline behavior.

In [ ]:
figure_paths = {label: repo_root / Path(raw_path) for label, raw_path in test_summary["representative_figures"].items()}
figure_paths["regression"] = repo_root / Path(test_summary["regression_figure"])

for label, figure_path in figure_paths.items():
    display(Markdown(f"### {label.title()}"))
    display(Image(filename=str(figure_path)))

The last cell opens the saved representative figures that the written report already points to. Seeing the best, median, worst, and regression cases side by side keeps the review honest: we can check where the method is strong, where it is merely acceptable, and where it still breaks down. That mix of cases is usually more informative than another round of aggregate statistics.